In [ ]:
import torch
from itertools import combinations
import time

In [ ]:
def fit_all_pairs_vectorized(X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:  
    G = X.T @ X
    g = X.T @ y
    sx = X.sum(dim=0)
    sy = y.sum()

    pairs = torch.tensor(list(combinations(range(X.shape[1]), 2)), dtype=torch.long, device=X.device)
    i, j = pairs[:, 0], pairs[:, 1]
    c = pairs.shape[0]

    A = torch.zeros((c, 3, 3), dtype=X.dtype, device=X.device)
    A[:, 0, 0] = float(X.shape[0],)
    A[:, 0, 1] = sx[i]
    A[:, 0, 2] = sx[j]
    A[:, 1, 0] = sx[i]
    A[:, 1, 1] = G[i, i]
    A[:, 1, 2] = G[i, j]
    A[:, 2, 0] = sx[j]
    A[:, 2, 1] = G[j, i]
    A[:, 2, 2] = G[j, j]

    b = torch.zeros((c, 3), dtype=X.dtype, device=X.device)
    b[:, 0] = sy
    b[:, 1] = g[i]
    b[:, 2] = g[j]

    betas = torch.linalg.lstsq(A, b)
    return betas.solution

def run_experiment(n, device='cpu', cols=100):
    X = torch.randn(n, cols, device=device)
    y = torch.randn(n, device=device)
    st = time.perf_counter()
    fit_all_pairs_vectorized(X, y)
    et = time.perf_counter()
    print(f"Time taken: {et - st:.3f}s")
    return  et - st

# CPU

In [ ]:
times = []
for n in [100_000, 1_000_000, 10_000_000]:
    t = run_experiment(n, device='cpu', cols=100)
    times.append(t)
    print("-" * 50)

# GPU

In [ ]:
times = []
for n in [100_000, 1_000_000, 10_000_000]:
    t = run_experiment(n, device='cuda', cols=100)
    torch.cuda.empty_cache()
    times.append(t)
    print("-" * 50)

# VRAM overflow

In [ ]:
def rows_for_GB(target_gb: float, cols: int, dtype: torch.dtype) -> int:
    bytes_per_element = torch.tensor([], dtype=dtype).element_size()
    bytes_total = target_gb * 1024**3
    return int(bytes_total // (cols * bytes_per_element))

In [ ]:
rows = rows_for_GB(7.5, 100, torch.float32)
print(rows)
X = torch.randn(rows, 100)
y = torch.randn(rows)

In [ ]:
fit_all_pairs_vectorized(X.to('cuda'), y.to('cuda'))
torch.cuda.empty_cache()

# Incremental Matrix multiply

In [ ]:
import torch
from itertools import combinations

def _gpu_gram_batch_rows(p: int, element_size: int, memory_fraction: float) -> int:
    """Max host rows to copy per chunk; heuristic from ols.PairwiseLinearRegression."""
    free, _ = torch.cuda.mem_get_info()
    max_bytes = int(free * memory_fraction)
    return max(1, max_bytes // (3 * p * element_size))


def incremental_gram_host_to_cuda(
    X: torch.Tensor,
    y: torch.Tensor,
    cuda_device: torch.device,
    memory_fraction: float = 0.9,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Accumulate G, g, sx, sy on CUDA while X and y remain on CPU."""
    if X.device.type != "cpu" or y.device.type != "cpu":
        raise ValueError("incremental_gram_host_to_cuda expects X and y on CPU")
    n, p = X.shape[0], X.shape[1]
    dtype = X.dtype
    G = torch.zeros((p, p), dtype=dtype, device=cuda_device)
    g = torch.zeros((p,), dtype=dtype, device=cuda_device)
    sx = torch.zeros((p,), dtype=dtype, device=cuda_device)
    sy = torch.zeros((), dtype=dtype, device=cuda_device)
    element_size = X.element_size()
    start = 0
    while start < n:
        batch_rows = _gpu_gram_batch_rows(p, element_size, memory_fraction)
        batch_rows = min(batch_rows, n - start)
        end = start + batch_rows
        Xb = X[start:end].to(cuda_device, non_blocking=True)
        yb = y[start:end].to(cuda_device, non_blocking=True)
        G += Xb.T @ Xb
        g += Xb.T @ yb
        sx += Xb.sum(dim=0)
        sy += yb.sum()
        start = end
        torch.cuda.empty_cache()
    return G, g, sx, sy

def fit_all_pairs_vectorized_incremental(X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    if X.device.type != 'cpu':
        raise Exception("X must live in RAM for incremental matrix multiply")

    G, g, sx, sy = incremental_gram_host_to_cuda(X, y, 'cuda', memory_fraction=0.9)

    pairs = torch.tensor(list(combinations(range(X.shape[1]), 2)), dtype=torch.long, device='cuda')
    i, j = pairs[:, 0], pairs[:, 1]
    c = pairs.shape[0]

    A = torch.zeros((c, 3, 3), dtype=X.dtype, device='cuda')
    A[:, 0, 0] = float(X.shape[0])
    A[:, 0, 1] = sx[i]
    A[:, 0, 2] = sx[j]
    A[:, 1, 0] = sx[i]
    A[:, 1, 1] = G[i, i]
    A[:, 1, 2] = G[i, j]
    A[:, 2, 0] = sx[j]
    A[:, 2, 1] = G[j, i]
    A[:, 2, 2] = G[j, j]

    b = torch.zeros((c, 3), dtype=X.dtype, device='cuda')
    b[:, 0] = sy
    b[:, 1] = g[i]
    b[:, 2] = g[j]

    betas = torch.linalg.lstsq(A, b)
    return betas.solution.cpu()

In [ ]:
rows = rows_for_GB(7.5, 100, torch.float32)
print(rows)
X = torch.randn(rows, 100)
y = torch.randn(rows)

In [ ]:
fit_all_pairs_vectorized_incremental(X, y)
torch.cuda.empty_cache()